# Angular Steering


This notebook contains:

- The process of extracting the refusal direction and constructing the steering plane.
- Visualization of the activation, extracted directions and constructed steering planes.
- The creation of the steering config that can be used with our fork of vLLM.

This notebook is inspired by https://colab.research.google.com/drive/1a-aQvKC9avdZpdyBn4jgRQFObTPy1JZw


## Setup


### Dependencies


In [ ]:
%%capture
!pip install transformers transformers_stream_generator tiktoken transformer_lens einops jaxtyping colorama nbformat plotly datasets pandas scikit-learn matplotlib

In [ ]:
import torch
import functools
import einops
import requests
import pandas as pd
import io
import textwrap
import gc
import numpy as np
import plotly
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from pathlib import Path
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch import Tensor
from typing import List
from transformer_lens import HookedTransformer, utils, ActivationCache
from transformer_lens.hook_points import HookPoint
from transformers import AutoTokenizer
from jaxtyping import Float, Int
from colorama import Fore
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt

In [ ]:
import os

# Only use GPU 0 and 2
os.environ["CUDA_VISIBLE_DEVICES"] = "5"

# Must set BEFORE importing torch

print(torch.cuda.device_count())  # should show 2

### Models and configs


In [ ]:
# Choose one model for the experimentw
MODEL_PATH = (
    # "Qwen/Qwen2.5-3B-Instruct"
    "Qwen/Qwen2.5-7B-Instruct"
    # "Qwen/Qwen2.5-14B-Instruct"
    # "Qwen/Qwen2.5-32B-Instruct"
    # "meta-llama/Llama-3.2-3B-Instruct"
    # "meta-llama/Llama-3.1-8B-Instruct"
    # "google/gemma-2-9b-it" 
    # "openai/gpt-oss-20b
    # "google/gemma-2b"
)



# METHOD_PREFIX = "PID_"
# METHOD_SFX = "_PID"

# METHOD_PREFIX = "RePE_"
# METHOD_SFX = "_RePE"

# METHOD_PREFIX = "ITI_"
# METHOD_SFX = "_ITI"

METHOD_PREFIX = ""
METHOD_SFX = ""

MODEL_NAME = MODEL_PATH.split("/")[-1]

DEVICE = "cuda:0"
BATCH_SIZE = 16

OUTPUT_DIR = Path("output") / f"{MODEL_NAME}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VISUALIZATION_DIR = Path("visualization") / f"{MODEL_NAME}"
VISUALIZATION_DIR.mkdir(parents=True, exist_ok=True)

from transformer_lens.loading_from_pretrained import OFFICIAL_MODEL_NAMES

if MODEL_PATH not in OFFICIAL_MODEL_NAMES:
    OFFICIAL_MODEL_NAMES.append(MODEL_PATH)

model = HookedTransformer.from_pretrained_no_processing(
    MODEL_PATH,
    device=DEVICE,
    dtype=torch.bfloat16,
    default_padding_side="left",
    # bf16=True
)

model.tokenizer.padding_side = "left"

# store original chat template
ORIGINAL_CHAT_TEMPLATE = model.tokenizer.chat_template

In [ ]:
# handle pad token for some model
if not model.tokenizer.pad_token:
    if "qwen1" in MODEL_PATH.lower():
        model.tokenizer.pad_token = "<|endoftext|>"
    elif model.tokenizer.eos_token:
        model.tokenizer.pad_token = model.tokenizer.eos_token
    else:
        raise ValueError("No pad token found in the tokenizer.")

### Load harmful / harmless datasets


In [ ]:
def get_harmful_instructions():
    url = "https://raw.githubusercontent.com/llm-attacks/llm-attacks/main/data/advbench/harmful_behaviors.csv"
    response = requests.get(url)

    dataset = pd.read_csv(io.StringIO(response.content.decode("utf-8")))
    instructions = dataset["goal"].tolist()

    train, test = train_test_split(instructions, test_size=0.2, random_state=42)
    return train, test


def get_harmless_instructions():
    hf_path = "tatsu-lab/alpaca"
    dataset = load_dataset(hf_path)

    # filter for instructions that do not have inputs
    instructions = []
    for i in range(len(dataset["train"])):
        if dataset["train"][i]["input"].strip() == "":
            instructions.append(dataset["train"][i]["instruction"])

    train, test = train_test_split(instructions, test_size=0.2, random_state=42)
    return train, test


def get_harmful_instructions_jp():
    huggingface_id = "kunishou/do-not-answer-ja"
    dataset = load_dataset(huggingface_id)

    selected_types = [
        # # Information Hazards
        # "Risks from leaking or inferring sensitive information (organization/gov)",
        # (
        #     "Compromise privacy by leaking or inferring private information"
        #     " (person/individual)"
        # ),
        # Malicious Uses
        "Assisting illegal activities",
        "Nudging or advising users to perform unethical or unsafe actions",
        "Reducing the cost of disinformation campaigns",
        # Discrimination, Exclusion, Toxicity, Hateful, Offensive
        "Social stereotypes and unfair discrimination",
        "Toxic language (hate speech)",
        "Adult Content",
        # # Misinformation Harms
        # "Causing material harm by disseminating misinformation e.g. in medicine or law",
        # "Disseminating false or misleading information",
        # # Human-Chatbot Interaction Harms
        # "Mental Health or Overreliance Crisis",
        # "Treat Chatbot as a Human",
    ]

    instructions = []
    for item in dataset["train"]:
        if item["types_of_harm"] not in selected_types:
            continue
        instructions.append(item["question"])

    train, test = train_test_split(instructions, test_size=0.2, random_state=42)
    return train, test


def get_harmless_instructions_jp():
    huggingface_id = "Lazycuber/alpaca-jp"
    dataset = load_dataset(huggingface_id)

    # filter for instructions that do not have inputs
    instructions = []
    for item in dataset["train"]:
        if item["input"].strip() != "":
            continue
        inst = item["instruction"]
        inst = inst.strip("「」'")
        instructions.append(inst)

    train, test = train_test_split(instructions, test_size=0.2, random_state=42)
    return train, test

In [ ]:
LANGUAGE = "en"

if LANGUAGE == "en":
    harmful_inst_train, harmful_inst_test = get_harmful_instructions()
    harmless_inst_train, harmless_inst_test = get_harmless_instructions()
elif LANGUAGE == "jp":
    harmful_inst_train, harmful_inst_test = get_harmful_instructions_jp()
    harmless_inst_train, harmless_inst_test = get_harmless_instructions_jp()

print(f"Train: {len(harmful_inst_train)} harmful, {len(harmless_inst_train)} harmless")
print(f"Test: {len(harmful_inst_test)} harmful, {len(harmless_inst_test)} harmless")

In [ ]:
print("Harmful instructions:")
for i in range(4):
    print(f"\t{harmful_inst_train[i]}")
print("Harmless instructions:")
for i in range(4):
    print(f"\t{harmless_inst_train[i]}")

### Tokenization utils


In [ ]:
def instructions_to_chat_tokens(
    tokenizer: AutoTokenizer,
    instructions: List[str],
) -> Int[Tensor, "batch_size seq_len"]:
    if tokenizer.chat_template:
        convos = [
            [{"role": "user", "content": instruction}] for instruction in instructions
        ]
        return tokenizer.apply_chat_template(
            convos,
            padding=True,
            truncation=False,
            add_generation_prompt=True,
            return_tensors="pt",
        )
    else:
        return tokenizer(
            instructions, padding=True, truncation=False, return_tensors="pt"
        ).input_ids

In [ ]:
harmful_sample_toks = instructions_to_chat_tokens(
    tokenizer=model.tokenizer, instructions=harmful_inst_train[:2]
)
harmless_sample_toks = instructions_to_chat_tokens(
    tokenizer=model.tokenizer, instructions=harmless_inst_train[:2]
)

for sample in harmful_sample_toks[:2]:
    print(model.tokenizer.decode(sample))
    print("-" * 50)
for sample in harmless_sample_toks[:2]:
    print(model.tokenizer.decode(sample))
    print("-" * 50)

### Generation utils


In [ ]:
def _generate_with_hooks(
    model: HookedTransformer,
    toks: Int[Tensor, "batch_size seq_len"],
    max_tokens_generated: int = BATCH_SIZE,
    fwd_hooks=[],
) -> List[str]:

    all_toks = torch.zeros(
        (toks.shape[0], toks.shape[1] + max_tokens_generated),
        dtype=torch.long,
        device=toks.device,
    )
    all_toks[:, : toks.shape[1]] = toks

    for i in range(max_tokens_generated):
        with model.hooks(fwd_hooks=fwd_hooks):
            logits = model(all_toks[:, : -max_tokens_generated + i])
            next_tokens = logits[:, -1, :].argmax(
                dim=-1
            )  # greedy sampling (temperature=0)
            all_toks[:, -max_tokens_generated + i] = next_tokens

    return model.tokenizer.batch_decode(
        all_toks[:, toks.shape[1] :], skip_special_tokens=False
    )


def get_generations(
    model: HookedTransformer,
    instructions: List[str],
    tokenizer: AutoTokenizer,
    fwd_hooks=[],
    max_tokens_generated: int = 64,
    batch_size: int = BATCH_SIZE,
) -> List[str]:

    generations = []

    for i in tqdm(range(0, len(instructions), batch_size)):
        toks = instructions_to_chat_tokens(
            tokenizer=tokenizer, instructions=instructions[i : i + batch_size]
        )

        with torch.no_grad():
            generation = _generate_with_hooks(
                model,
                toks,
                max_tokens_generated=max_tokens_generated,
                fwd_hooks=fwd_hooks,
            )
        generations.extend(generation)

    return generations

In [ ]:
def run_single_sample(model, input, tokenizer, fwd_hooks=[], max_tokens_generated=64):
    baseline_generations = get_generations(
        model,
        [input],
        tokenizer,
        fwd_hooks=[],
        max_tokens_generated=max_tokens_generated,
    )
    intervention_generations = get_generations(
        model,
        [input],
        tokenizer,
        fwd_hooks=fwd_hooks,
        max_tokens_generated=max_tokens_generated,
    )

    print(f"INSTRUCTION: {repr(input)}")
    print(Fore.GREEN + f"BASELINE COMPLETION:")
    print(
        textwrap.fill(
            baseline_generations[0],
            width=100,
            initial_indent="\t",
            subsequent_indent="\t",
        )
    )
    print(Fore.RED + f"INTERVENTION COMPLETION:")
    print(
        textwrap.fill(
            intervention_generations[0],
            width=100,
            initial_indent="\t",
            subsequent_indent="\t",
        )
    )

## Finding the "refusal direction"


### Helper functions


In [ ]:
def __run_with_cache(model, data, batch_size):
    cache = {}
    with torch.no_grad():
        for i in range(0, len(data), batch_size):
            _, batch_cache = model.run_with_cache(
                data[i : i + batch_size],
                names_filter=lambda hook_name: "resid" in hook_name,
                return_cache_object=False,
            )
            for k, v in batch_cache.items():
                if k not in cache:
                    cache[k] = v.cpu()
                else:
                    cache[k] = torch.vstack([cache[k], v.cpu()])

    return ActivationCache(cache, model)

In [ ]:
def get_template_suffix_toks(tokenizer):
    # Since the padding is on the left side, the suffix of all samples are the same
    # when using the same template.
    # The activations on these suffix tokens are after the prompt has been processed,
    # thus it's interesting to see how the activations differ between contrastive
    # samples

    # get the common suffix between 2 samples
    toks = instructions_to_chat_tokens(tokenizer=tokenizer, instructions=["a", "b"])
    suffix = toks[0]
    for i in range(len(toks[0]) - 1, -1, -1):
        if toks[0][i] != toks[1][i]:
            suffix = toks[0][i + 1 :]

    return tokenizer.convert_ids_to_tokens(suffix)

In [ ]:
def get_activations(
    model: HookedTransformer,
    instructions: List[str],
    batch_size: int = BATCH_SIZE,
    act_names: List[str] = ["resid_mid", "resid_post"],
    num_last_tokens: int = 1,
):
    # tokenize instructions
    toks = instructions_to_chat_tokens(
        tokenizer=model.tokenizer, instructions=instructions
    )
    # toks.to(model.device())
    # run model on instructions and cache activations
    with torch.no_grad():
        cache = __run_with_cache(model, toks, batch_size=BATCH_SIZE)

    # get activations for the last n tokens
    acts = torch.stack(
        [
            torch.stack(
                [cache[act, layer][:, -num_last_tokens:, :] for act in act_names]
            )
            for layer in range(model.cfg.n_layers)
        ]
    )

    # layers x resid_modules x batch x tokens x dim
    return acts, cache

### Extract the activations


In [ ]:
N_INST_TRAIN = 512
BATCH_SIZE = 32

# extraction points per decoder block
act_names = ["resid_mid", "resid_post"]

# get the template suffix tokens
template_suffix_toks = get_template_suffix_toks(model.tokenizer)
if not template_suffix_toks:
    template_suffix_toks = ["<last token>"]

# only get the activations of the template suffix tokens since these tokens are the same
# for all samples
num_last_tokens = len(template_suffix_toks)
print("template_suffix_toks:", template_suffix_toks)

# load from cache if exists
output_file = OUTPUT_DIR / f"acts_harmful_{LANGUAGE}_{MODEL_PATH.split('/')[-1]}.npy"
if output_file.exists():
    print("Loading harmful activations from file")
    harmful_acts = np.load(output_file)
    harmful_acts = torch.from_numpy(harmful_acts)
else:
    # get activations for harmful instructions then save to file
    harmful_acts, cache = get_activations(
        model,
        harmful_inst_train[:N_INST_TRAIN],
        batch_size=BATCH_SIZE,
        act_names=act_names,
        num_last_tokens=num_last_tokens,
    )
    harmful_acts = harmful_acts.cpu().float()
    np.save(output_file, harmful_acts.numpy())

# load from cache if exists
output_file = OUTPUT_DIR / f"acts_harmless_{LANGUAGE}_{MODEL_PATH.split('/')[-1]}.npy"
if output_file.exists():
    print("Loading harmless activations from file")
    harmless_acts = np.load(output_file)
    harmless_acts = torch.from_numpy(harmless_acts)
else:
    # get activations for harmless instructions then save to file
    harmless_acts, cache = get_activations(
        model,
        harmless_inst_train[:N_INST_TRAIN],
        batch_size=BATCH_SIZE,
        act_names=act_names,
        num_last_tokens=num_last_tokens,
    )
    harmless_acts = harmless_acts.cpu().float()
    np.save(output_file, harmless_acts.numpy())

In [ ]:
import torch

def median_of_means(tensor: torch.Tensor, num_blocks: int, dim: int) -> torch.Tensor:
    """
    Compute the Median of Means along a specified dimension.

    Args:
        tensor (torch.Tensor): Input tensor of any shape.
        num_blocks (int): Number of blocks to divide the data along `dim`.
        dim (int): The dimension along which to compute the MoM.

    Returns:
        torch.Tensor: Tensor with same shape as input, but reduced over `dim`.
    """
    N = tensor.size(dim)
    if num_blocks <= 0 or num_blocks > N:
        raise ValueError(f"num_blocks must be between 1 and {N}")

    # Move the target dimension to the front
    perm = [dim] + [i for i in range(tensor.ndim) if i != dim]
    tensor_perm = tensor.permute(perm)  # Now shape (N, ...)

    shape = tensor_perm.shape
    tensor_flat = tensor_perm.reshape(N, -1)  # Flatten rest: (N, M)

    # Shuffle along N
    idx = torch.randperm(N, device=tensor.device)
    shuffled = tensor_flat[idx]

    # Determine block sizes
    block_sizes = [N // num_blocks + (1 if i < (N % num_blocks) else 0) for i in range(num_blocks)]

    block_means = []
    start = 0
    for size in block_sizes:
        end = start + size
        block = shuffled[start:end]  # (block_size, M)
        block_mean = block.mean(dim=0)  # (M,)
        block_means.append(block_mean)
        start = end

    block_means_tensor = torch.stack(block_means, dim=0)  # (num_blocks, M)
    mom_flat = block_means_tensor.median(dim=0).values  # (M,)

    # Reshape back to shape without the reduced dim
    reduced_shape = [tensor.shape[i] for i in range(tensor.ndim) if i != dim]
    mom = mom_flat.view(*reduced_shape)

    return mom


### Analyze the activations


In [ ]:
from torch.nn.functional import cosine_similarity, normalize


def get_pairwise_cosine_similarity(acts_normed):
    # comput cosine similarity of each pair of vector from a set of normalized vectors
    # acts_normed is ... x batch x toks x dim

    acts_normed = torch.tensor(acts_normed, device="cuda:0")

    # ... batch1 toks dim, ... batch2 toks dim -> ... toks batch1 batch2
    acts_pairwise_sim = torch.einsum("...ikl,...jkl->...kij", acts_normed, acts_normed)

    batch_size = acts_pairwise_sim.shape[-1]

    # get the indices of the upper triangular part of the batch x batch similarity matrix
    indices = np.arange(batch_size**2).reshape(batch_size, batch_size)
    indices = indices[np.triu_indices_from(indices, k=1)]

    # ... x toks x batch x (batch * batch)
    acts_pairwise_sim = acts_pairwise_sim.reshape(*acts_pairwise_sim.shape[:-2], -1)
    # ... x toks x batch x (batch * (batch - 1) // 2)
    acts_pairwise_sim = acts_pairwise_sim[..., indices]
    # ... x (batch * (batch - 1) // 2) x toks
    acts_pairwise_sim = acts_pairwise_sim.swapaxes(-1, -2)

    return acts_pairwise_sim


def get_cosine_with_mean(acts_normed):
    # compute cosine similarity of each vector with the mean vector
    # acts_normed is ... x batch x toks x dim

    acts_normed = torch.tensor(acts_normed, device="cuda:0")
    mean_act = acts_normed.mean(axis=2)
    mean_act /= mean_act.norm(dim=-1, keepdim=True)

    # ... batch toks dim, ... toks dim -> ... batch toks
    cosine_with_mean = torch.einsum("...ijk,...jk ->...ij", acts_normed, mean_act)

    return cosine_with_mean


# layers x resid_modules x batch x tokens x dim
harmful_acts_normed = harmful_acts / harmful_acts.norm(dim=-1, keepdim=True)
harmless_acts_normed = harmless_acts / harmless_acts.norm(dim=-1, keepdim=True)

# shape: layers x resid_modules x tokens x dim
# normalize then get mean because the activation will be normalized by the RMSNorm layer
# normalize helps to preserve the directions as the magnitudes are irrelevant after the
# RMSNorm layer

print(harmful_acts_normed.shape)
harmful_acts_normed_mean = harmful_acts_normed.mean(dim=2)
harmless_acts_normed_mean = harmless_acts_normed.mean(dim=2)

harmful_acts_normed_mom = median_of_means(harmful_acts_normed, num_blocks=8, dim=2)
harmless_acts_normed_mom = median_of_means(harmless_acts_normed, num_blocks=8, dim=2)
# layers x resid_modules x tokens
similarity_scores = (
    cosine_similarity(harmful_acts_normed_mean, harmless_acts_normed_mean, dim=-1)
    .cpu()
    .float()
    .numpy()
)


hidden_dim = harmful_acts.shape[-1]
# shape: layers x resid_modules x tokens x dim
# Rescale the activations to the same as in RMSNorm (sqrt(hidden_dim))
# This effectively makes the values of each vector be standard normal
# So regardless of the hidden dimension, each vector will always be a sample from
# standard normal
# Hence the variance of activation values will be 1
harmful_acts_normed_var = (
    (harmful_acts_normed * np.sqrt(hidden_dim)).var(dim=2).cpu().float().numpy()
)
harmless_acts_normed_var = (
    (harmless_acts_normed * np.sqrt(hidden_dim)).var(dim=2).cpu().float().numpy()
)

harmful_acts_normed = harmful_acts_normed.cpu().float().numpy()
harmless_acts_normed = harmless_acts_normed.cpu().float().numpy()


# layers x resid_modules x batch x tokens
# cosine of each vector with the mean vector
harmful_acts_cosine_with_mean = get_cosine_with_mean(harmful_acts_normed).cpu().numpy()
harmless_acts_cosine_with_mean = (
    get_cosine_with_mean(harmless_acts_normed).cpu().numpy()
)

# layers x resid_modules x (batch * (batch - 1) // 2) x tokens
# cosine similarity of each pair of vectors
harmful_acts_pairwise_sim = (
    get_pairwise_cosine_similarity(harmful_acts_normed).cpu().numpy()
)
harmless_acts_pairwise_sim = (
    get_pairwise_cosine_similarity(harmless_acts_normed).cpu().numpy()
)

# layers x resid_modules x tokens
# variance of cosine similarity of each pair of vectors
harmful_acts_pairwise_sim_var = np.var(harmful_acts_pairwise_sim, axis=-2)
harmless_acts_pairwise_sim_var = np.var(harmless_acts_pairwise_sim, axis=-2)

acts_normed_var = dict()

# layers x resid_modules x tokens
acts_normed_var["harmful"] = dict(
    mean=harmful_acts_normed_var.mean(axis=-1),
    max=harmful_acts_normed_var.max(axis=-1),
)

# layers x resid_modules x tokens
acts_normed_var["harmless"] = dict(
    mean=harmless_acts_normed_var.mean(axis=-1),
    max=harmless_acts_normed_var.max(axis=-1),
)

In [ ]:
# clean up memory
gc.collect()
torch.cuda.empty_cache()

### Visualize the activations


#### Cosine Similarity between harmful and harmless activations at each layer and token position


In [ ]:
num_layers, num_act_modules, num_tokens = similarity_scores.shape
data = similarity_scores.reshape(-1, similarity_scores.shape[-1])
y_labels = sum([[f"{layer}-mid", f"{layer}-post"] for layer in range(num_layers)], [])
x_labels = [repr(tok) for tok in template_suffix_toks]


fig = px.imshow(
    data,
    y=y_labels,
    labels={"x": "token position", "y": "layer", "color": "cosine similarity"},
    aspect="auto",
)
fig.update_layout(
    xaxis={
        "tickmode": "array",
        "ticktext": x_labels,
        "tickvals": list(range(len(x_labels))),
    },
    yaxis={
        "tickmode": "array",
        "ticktext": list(range(len(y_labels))),
        "tickvals": list(range(0, len(y_labels), len(act_names))),
    },
    title=(
        "Cosine Similarity between harmful and harmless activations at each layer and"
        " token position"
    ),
)
fig.show()

In [ ]:
def variance_plot(**kwargs):
    x = kwargs.pop("x")
    y = kwargs.pop("y")
    y_mean = y.mean(dim=-1)
    y_std = y.std(dim=-1)
    y_upper = y_mean + y_std
    y_lower = y_mean - y_std
    y_upper = y_upper.tolist()
    y_lower = y_lower.tolist()
    # colour = kwargs.pop("color")

    trace = go.Scatter(
        x=x + x[::-1],
        y=y_upper + y_lower[::-1],
        mode="lines",
        fill="toself",
        line=dict(color=kwargs["fillcolor"], width=0),
        **kwargs
    )

    return trace

#### Activation norms at each extraction point


In [ ]:
num_layers, num_act_modules, num_tokens = similarity_scores.shape

chosen_token = -1
colour_map = {
    "harmless": plotly.colors.qualitative.Plotly[0],
    "harmful": plotly.colors.qualitative.Plotly[1],
    "neutral": plotly.colors.qualitative.Plotly[3],
}
colour_map_light = {
    "harmless": plotly.colors.qualitative.Pastel1[1],
    "harmful": plotly.colors.qualitative.Pastel1[0],
    "neutral": plotly.colors.qualitative.Pastel1[3],
}
colour_map_opaque = {
    "harmless": None,
    "harmful": "rgba(251, 180, 174, 0.3)",
    "harmless": "rgba(179, 205, 227, 0.3)",
}

# layers x resid_modules x tokens x batch x dim
acts = {"harmful": harmful_acts, "harmless": harmless_acts}

categories = ["harmless", "harmful"]
resid_modules = ["mid", "post"]

x_values = sum([[f"{l}", f"{l}-post"] for l in range(num_layers)], [])
x_values = [str(i) for i in range(2 * num_layers)]

fig = go.Figure()

for category in categories:
    normed_acts = acts[category].norm(dim=-1)
    mean_normed_acts = normed_acts.mean(dim=-1)

    y_values = mean_normed_acts[..., chosen_token].flatten()

    # mean
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=y_values,
            name=category,
            mode="lines+markers",
            yaxis="y",
            marker=dict(color=colour_map[category], size=3),
            showlegend=True,
        )
    )
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=y_values,
            mode="lines+markers",
            yaxis="y",
            marker=dict(color=colour_map_light[category], size=3),
            showlegend=False,
        )
    )

    # variance
    fig.add_trace(
        variance_plot(
            x=x_values,
            y=normed_acts[:, :, chosen_token].reshape(-1, normed_acts.shape[-1]),
            yaxis="y",
            fillcolor=colour_map_opaque[category],
            showlegend=False,
        )
    )

    # dot markers
    fig.add_trace(
        go.Scatter(
            x=x_values[1::],
            y=y_values[1::],
            name=f"{category}",
            mode="markers",
            yaxis="y",
            marker=dict(color=colour_map[category], size=3),
            showlegend=False,
        )
    )
    # for module_idx, module_name in enumerate(resid_modules):
    #     if module_name == "post":
    #         colour = colour_map[category]
    #     else:
    #         colour = colour_map_light[category]
    #     fig.add_trace(
    #         go.Scatter(
    #             x=x_values[module_idx::2],
    #             y=y_values[module_idx::2],
    #             name=f"{category}-{module_name}",
    #             mode="markers",
    #             yaxis="y",
    #             marker=dict(color=colour, size=3),
    #             showlegend=True,
    #         )
    #     )


fig.update_layout(
    # title=f"Activation norms at each layer for {MODEL_PATH}",
    plot_bgcolor="white",
    grid=dict(rows=1, columns=1),
    xaxis=dict(
        type="category",
        dtick=4,
        title=dict(text="Extraction Point", font=dict(size=20)),
        gridcolor="lightgrey",
        tickfont=dict(size=18),
    ),
    yaxis=dict(
        title=dict(text="Activation Norm", font=dict(size=20)),
        gridcolor="lightgrey",
        zeroline=False,
        tickfont=dict(size=18),
    ),
    hovermode="x unified",
    height=250,
    # width=20 + 12 * len(x_values),
    width=600,
    margin=dict(l=0, r=0, t=0, b=0),
    legend=dict(x=0.05, y=0.95, font=dict(size=18)),
)
fig.show()

fig.write_image(VISUALIZATION_DIR / "acts_norm.pdf", scale=5)

#### Variance of normed activations at each extraction point


In [ ]:
num_layers, num_act_modules, num_tokens = similarity_scores.shape

chosen_token = -1
colour_map = {
    "harmless": plotly.colors.qualitative.Plotly[0],
    "harmful": plotly.colors.qualitative.Plotly[1],
    "neutral": plotly.colors.qualitative.Plotly[3],
}
colour_map_light = {
    "harmless": plotly.colors.qualitative.Pastel1[1],
    "harmful": plotly.colors.qualitative.Pastel1[0],
    "neutral": plotly.colors.qualitative.Pastel1[3],
}

categories = ["harmless", "harmful"]
resid_modules = ["mid", "post"]
metrics = ["mean", "max"]

x_values = sum([[f"{l}", f"{l}-post"] for l in range(num_layers)], [])

fig = go.Figure()

for m, metric in enumerate(metrics):
    for category in categories:
        y_values = acts_normed_var[category][metric][..., chosen_token]
        fig.add_trace(
            go.Scatter(
                x=x_values,
                y=y_values.flatten(),
                mode="lines",
                yaxis=f"y{m + 1}",
                marker=dict(color=colour_map_light[category], size=5),
                showlegend=False,
            )
        )
        for module_idx, module_name in enumerate(resid_modules):
            if module_name == "mid":
                colour = colour_map[category]
            else:
                colour = colour_map_light[category]
            fig.add_trace(
                go.Scatter(
                    x=x_values[module_idx::2],
                    y=y_values.flatten()[module_idx::2],
                    name=f"{category}-{module_name}",
                    mode="markers",
                    yaxis=f"y{m + 1}",
                    marker=dict(color=colour, size=5),
                    showlegend=m == 0,
                )
            )

diff_mean_var = (
    acts_normed_var["harmless"]["mean"][..., chosen_token]
    - acts_normed_var["harmful"]["mean"][..., chosen_token]
)
fig.add_trace(
    go.Scatter(
        x=x_values,
        y=y_values.flatten(),
        mode="lines",
        yaxis="y3",
        marker=dict(color=colour_map_light["neutral"], size=5),
        showlegend=False,
    )
)
for module_idx, module_name in enumerate(resid_modules):
    if module_name == "mid":
        colour = colour_map["neutral"]
    else:
        colour = colour_map_light["neutral"]
    fig.add_trace(
        go.Scatter(
            x=x_values[module_idx::2],
            y=y_values.flatten()[module_idx::2],
            name=f"(harmelss - harmful)-{module_name}",
            mode="markers",
            yaxis="y3",
            marker=dict(color=colour, size=5),
            showlegend=True,
        )
    )


fig.update_layout(
    title=f"Variance of normed activations at each layer for {MODEL_PATH}",
    plot_bgcolor="white",
    grid=dict(rows=3, columns=1),
    xaxis=dict(
        type="category", dtick=2, title="Transformers Block", gridcolor="lightgrey"
    ),
    yaxis=dict(title=f"{metrics[0]} variance", gridcolor="lightgrey", zeroline=False),
    yaxis2=dict(title=f"{metrics[1]} variance", gridcolor="lightgrey", zeroline=False),
    yaxis3=dict(title=f"harmless - harmful", gridcolor="lightgrey", zeroline=False),
    hovermode="x unified",
    height=1200,
    # width=20 + 12 * len(x_values),
)

# fig.update_xaxes(
#     mirror=True,
#     ticks='outside',
#     showline=True,
#     # linecolor='black',
#     gridcolor='lightgrey'
# )
# fig.update_yaxes(
#     mirror=True,
#     ticks='outside',
#     showline=True,
#     # linecolor='black',
#     gridcolor='lightgrey'
# )
fig.show()

#### Statistics of activations between harmful and harmless activations at each extraction point


In [ ]:
x_values = sum([[f"{l}", f"{l}-post"] for l in range(num_layers)], [])

fig = go.Figure()

# Angles between each pair of mean vectors at each layer
fig.add_trace(
    go.Scatter(
        x=x_values,
        y=np.arccos(similarity_scores[..., chosen_token]).flatten(),
        name="harmful-harmless angle",
        # showlegend=False,
        mode="lines+markers",
        marker=dict(color=colour_map_light["neutral"], size=5),
        yaxis="y",
    )
)
fig.add_trace(
    go.Scatter(
        x=x_values[1::2],
        y=np.arccos(similarity_scores[..., 1, chosen_token]).flatten(),
        # name="cosine similarity",
        showlegend=False,
        mode="markers",
        marker=dict(color=colour_map["neutral"], size=5),
        yaxis="y",
    )
)

for category in categories:
    if category == "harmful":
        # layers x resid_modules x (batch * (batch - 1) // 2)
        # cosine of each sample activation vector with other sample activation vectors
        acts_pairwise_sim = harmful_acts_pairwise_sim[..., chosen_token]

        # layers x resid_modules x batch
        # cosine of each sample activation vector with the mean activation vector
        acts_cosine_with_mean = harmful_acts_cosine_with_mean[..., chosen_token].clip(
            -1, 1
        )
    else:
        # layers x resid_modules x (batch * (batch - 1) // 2)
        # cosine of each sample activation vector with other sample activation vectors
        acts_pairwise_sim = harmless_acts_pairwise_sim[..., chosen_token]

        # layers x resid_modules x batch
        # cosine of each sample activation vector with the mean activation vector
        acts_cosine_with_mean = harmless_acts_cosine_with_mean[..., chosen_token].clip(
            -1, 1
        )

    acts_arccos_with_mean = np.arccos(acts_cosine_with_mean)

    count = acts_cosine_with_mean.shape[-1]

    for module_idx, module_name in enumerate(resid_modules):
        if module_name == "pre":
            colour = colour_map[category]
        else:
            colour = colour_map_light[category]

        # fig.add_trace(
        #     go.Box(
        #         x=sum([[name] * count for name in x_values[module_idx::2]], []),
        #         y=np.arccos(acts_arccos_with_mean[..., module_idx, :]).flatten(),
        #         boxmean=True,
        #         line_width=1,
        #         marker_size=2,
        #         showlegend=False,
        #         marker_color=colour,
        #         yaxis="y2",
        #     ),
        # )
        fig.add_trace(
            variance_plot(
                x=x_values,
                y=torch.tensor(acts_arccos_with_mean).reshape(
                    -1, acts_arccos_with_mean.shape[-1]
                ),
                yaxis="y2",
                fillcolor=colour_map_opaque[category],
                showlegend=False,
            )
        )

    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=acts_arccos_with_mean.mean(axis=-1).flatten(),
            mode="lines+markers",
            showlegend=False,
            marker=dict(color=colour_map[category], size=3),
            line_width=1,
            yaxis="y2",
        ),
    )
    # fig.add_trace(
    #     go.Scatter(
    #         x=x_values[1::2],
    #         y=acts_pairwise_sim.mean(axis=-1).flatten(),
    #         mode="markers",
    #         showlegend=False,
    #         marker_color=colour_map[category],
    #         yaxis="y4",
    #     ),
    # )


harmful_locality_scores = np.arccos(
    harmful_acts_cosine_with_mean[..., chosen_token].clip(-1, 1)
).mean(axis=-1)
harmless_locality_scores = np.arccos(
    harmless_acts_cosine_with_mean[..., chosen_token].clip(-1, 1)
).mean(axis=-1)
locality_scores = np.maximum(harmful_locality_scores, harmless_locality_scores)

sparsity_scores = np.arccos(similarity_scores[..., chosen_token].clip(-1, 1))

# scores = sparsity_scores * 2 / (harmful_locality_scores + harmless_locality_scores)
# scores = sparsity_scores - locality_scores
scores = sparsity_scores / locality_scores
# scores[
#     sparsity_scores <= np.minimum(harmful_locality_scores, harmless_locality_scores)
# ] = np.nan
fig.add_trace(
    go.Scatter(x=x_values, y=scores.flatten(), mode="lines+markers", yaxis="y3")
)


fig.update_layout(
    title=(
        "Statistics of activations between harmful and harmless activations at each"
        f" layer for {MODEL_PATH}"
    ),
    plot_bgcolor="white",
    grid=dict(rows=3, columns=1),
    xaxis=dict(
        type="category", title="Transformers Block", dtick=2, gridcolor="lightgrey"
    ),
    yaxis=dict(
        title="Harmful-Harmless angle (1)", gridcolor="lightgrey", zeroline=False
    ),
    yaxis2=dict(title="Pairwise arccos (2)", gridcolor="lightgrey", zeroline=False),
    yaxis3=dict(title="(1) / max((2))", gridcolor="lightgrey", zeroline=False),
    # yaxis4=dict(title="Pairwise Cosine Similarity between samples"),
    hovermode="x unified",
    height=1200,
    # width=20 + 12 * len(x_values),
)
fig.show()

# an adhoc attempt to find the best direction
chosen_layer, chosen_act_idx = np.unravel_index(
    np.nanargmax(scores, axis=None), scores.shape
)
print(
    f"Best direction at layer {chosen_layer}, module"
    f" {act_names[chosen_act_idx]}, position {chosen_token}"
)

In [ ]:
# another adhoc attempt to find the best direction

harmful_locality_scores = np.arccos(harmful_acts_cosine_with_mean.clip(-1, 1)).mean(
    axis=-2
)
harmless_locality_scores = np.arccos(harmless_acts_cosine_with_mean.clip(-1, 1)).mean(
    axis=-2
)
locality_scores = harmful_locality_scores + harmless_locality_scores

sparsity_scores = np.arccos(similarity_scores.clip(-1, 1))

scores = sparsity_scores / np.maximum(harmful_locality_scores, harmless_locality_scores)
print(scores.shape)
# scores[
#     sparsity_scores <= np.minimum(harmful_locality_scores, harmless_locality_scores)
# ] = np.nan
scores = scores[..., -2:]
_chosen_layer, _chosen_act_idx, _chosen_token = np.unravel_index(
    np.nanargmax(scores, axis=None), scores.shape
)
print(
    f"Lowest cosine similarity at layer {_chosen_layer}, module"
    f" {act_names[_chosen_act_idx]}, position {_chosen_token}"
)

### Store the candidate directions from every extraction point


In [ ]:
def PID_control(candidates: torch.Tensor, kp, ki, kd):
    num_cands = candidates.shape[0]
    new_candidates = torch.zeros_like(candidates, device=candidates.device)
    new_candidates[0] = candidates[0]
    for i in range(1, num_cands):
        new_candidates[i] = kp*candidates[i] + ki*(torch.sum(new_candidates[:i], dim=0) + candidates[i]) + kd*(candidates[i] - new_candidates[i-1])
    
    return new_candidates

In [ ]:
def RePE(candiates):
    # 1. Center each batch along samples
    candiates_c = candiates - np.mean(candiates, axis=2, keepdims=True) 

    # 2. Batched SVD
    # torch.linalg.svd supports batching
    _, _, Vh = np.linalg.svd(candiates_c, full_matrices=False)

    # 3. First principal component for each batch
    v1 = Vh[:, :, 0, :]   # shape: (B, D)
    return v1

In [ ]:
def ITI(X, y, num_layers):

    # --- model factory ---
    class LogisticLR(nn.Module):
        def __init__(self, in_dim):
            super().__init__()
            self.linear = nn.Linear(in_dim, 1, bias=False)  # logistic regression = linear + sigmoid
        def forward(self, x):
            return self.linear(x).squeeze(-1)   # logits

    def train_one(model, loader, epochs=5, lr=1e-2, device="cuda" if torch.cuda.is_available() else "cpu"):
        model.to(device)
        opt  = torch.optim.AdamW(model.parameters(), lr=lr)
        loss = nn.BCEWithLogitsLoss()
        model.train()
        l=10
        for e in range(epochs):
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                l = loss(logits, yb)
                opt.zero_grad(); l.backward(); opt.step()
        return model, l

    # --- initialize & train 32 models ---
    models = [LogisticLR(X.shape[-1]) for _ in range(num_layers)]
    for i, m in enumerate(models):
        print(f'==== Training Logistic Classifier for Layer {i} ====')
        torch.manual_seed(1234 + i)  # different init per model
        ds = TensorDataset(X[i], y)
        loader = DataLoader(ds, batch_size=64, shuffle=True)
        _, loss = train_one(m, loader, epochs=20)
        print(loss)
    return torch.stack([m.linear.weight.data.squeeze(0) for m in models])

In [ ]:
chosen_act_idx = np.argmin(similarity_scores[chosen_layer, :, chosen_token])
refusal_dirs_path = (
    OUTPUT_DIR
    / f"refusal_dirs_{chosen_token}_{LANGUAGE}_{MODEL_PATH.split('/')[-1]}{METHOD_SFX}.npy"
)

# This is where you can change how to compute intervention vectors
# load from cache if exists
if refusal_dirs_path.exists():
    print("loading refusal_dirs from file")
    refusal_dirs = np.load(refusal_dirs_path)
else:
    harmful_acts_normed_mean_normed = normalize(
        harmful_acts_normed_mean[:, :, chosen_token], dim=-1
    )
    harmless_acts_normed_mean_normed = normalize(
        harmless_acts_normed_mean[:, :, chosen_token], dim=-1
    )
    refusal_dirs = harmful_acts_normed_mean_normed - harmless_acts_normed_mean_normed
    print(refusal_dirs.shape)

    if "PID" in METHOD_PREFIX:
        prefix_sum = torch.cumsum(refusal_dirs, dim=0)
        shifted_refusal_dirs = refusal_dirs.roll(1, dims=0)
        shifted_refusal_dirs[0] = refusal_dirs[0]
        diff_from_first = refusal_dirs - shifted_refusal_dirs
        
        if "PID_" in METHOD_PREFIX:    
            refusal_dirs = 0.9*refusal_dirs + 0.01*prefix_sum + 0.01*diff_from_first
            # refusal_dirs = PID_control(refusal_dirs, kp=1, ki=0.1, kd=0.1)
        elif "PI_" in METHOD_PREFIX:
            # refusal_dirs = refusal_dirs + 0.05*prefix_sum 
            refusal_dirs = PID_control(refusal_dirs, kp=1, ki=0.1, kd=0.0)
        elif "PD_" in METHOD_PREFIX:
            # refusal_dirs = refusal_dirs + 0.1*diff_from_first
            refusal_dirs = PID_control(refusal_dirs, kp=1, ki=0.0, kd=0.1)
            
    elif "RePE" in METHOD_PREFIX:
        all_refusal_dirs_norm = harmful_acts_normed[:, :, :, chosen_token] - harmless_acts_normed[:, :, :harmful_acts_normed.shape[2], chosen_token]
        all_refusal_dirs_norm = all_refusal_dirs_norm / np.linalg.norm(all_refusal_dirs_norm, axis=-1,keepdims=True)
    
        refusal_dirs = torch.tensor(RePE(all_refusal_dirs_norm))
        print(refusal_dirs.shape)
    
    elif "ITI" in METHOD_PREFIX:
        all_refusal_dirs_norm = harmful_acts_normed[:, :, :, chosen_token] - harmless_acts_normed[:, :, :harmful_acts_normed.shape[2], chosen_token]
        X = torch.tensor(np.concatenate([harmful_acts_normed[:, :, :, chosen_token], harmless_acts_normed[:, :, :, chosen_token]],axis=2)).view(harmful_acts_normed.shape[0]*harmful_acts_normed.shape[1], -1, harmful_acts_normed.shape[-1])
        y = torch.cat([torch.ones(harmful_acts_normed.shape[2]), torch.zeros(harmless_acts_normed.shape[2])])
        print(X.shape, y.shape)
        refusal_dirs = ITI(X, y, X.shape[0])
        refusal_dirs = refusal_dirs.view(harmful_acts_normed.shape[0], harmful_acts_normed.shape[1], harmful_acts_normed.shape[-1]).to(all_refusal_dirs_norm.device) \
            * torch.tensor(all_refusal_dirs_norm.std(axis=2))
        print(refusal_dirs.shape)
    else:
        refusal_dirs = refusal_dirs
        
    # refusal_dirs /= refusal_dirs.norm(dim=-1, keepdim=True)
    refusal_dirs = refusal_dirs.cpu().float().numpy()
    print("save refusal_dirs to file")
    np.save(refusal_dirs_path, refusal_dirs)

# for sanity check
print(chosen_layer, chosen_act_idx, chosen_token)
_harmful_dir = harmful_acts_normed_mom[chosen_layer, chosen_act_idx, chosen_token]
_harmful_dir = normalize(_harmful_dir, dim=-1)
_harmless_dir = harmless_acts_normed_mom[chosen_layer, chosen_act_idx, chosen_token]
_harmless_dir = normalize(_harmless_dir, dim=-1)
_refusal_dir = _harmful_dir - _harmless_dir

print(_refusal_dir.shape)
_refusal_dir /= _refusal_dir.norm()
dtype = _refusal_dir.dtype

# sanity check
print(_refusal_dir.float())
print(torch.tensor(refusal_dirs[chosen_layer, chosen_act_idx]))

# assert np.allclose(_refusal_dir.float(), refusal_dirs[chosen_layer, chosen_act_idx])

In [ ]:
def plot_vector_norms(X: torch.Tensor, ord: float = 2.0):
    """
    Compute and plot norms of vectors along the last dimension
    for a tensor of shape (36, 2, 2048).

    Args:
        X (torch.Tensor): Input tensor with shape (36, 2, 2048).
        ord (float): Norm order (e.g., 2 for L2, 1 for L1).
    """
    if X.ndim != 3 or X.shape[0] != 36 or X.shape[1] != 2:
        raise ValueError(f"Expected X to have shape (36, 2, *), got {tuple(X.shape)}")

    # Compute vector norms along the last dimension
    norms = torch.linalg.vector_norm(X, ord=ord, dim=-1)  # shape: (36, 2)

    # Create an index for plotting
    idx = list(range(X.shape[0]))

    # Plot both channels
    plt.figure(figsize=(9, 5))
    plt.plot(idx, norms[:, 0].tolist(), label="channel 0")
    plt.plot(idx, norms[:, 1].tolist(), label="channel 1")
    plt.xlabel("Index (0..35)")
    plt.ylabel(f"L{ord} norm of 2048-dim vector")
    plt.title("Vector norms across first dimension")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Also return a DataFrame for inspection or saving
    df = pd.DataFrame({
        "index": idx,
        "norm_ch0": norms[:, 0].tolist(),
        "norm_ch1": norms[:, 1].tolist(),
    })
    return df

In [ ]:
df_norms = plot_vector_norms(torch.Tensor(refusal_dirs), ord=2.0)

In [ ]:
def get_direction_ablation_output_hook(layer, direction: Tensor):
    def hook_fn(output, hook):
        nonlocal direction

        if isinstance(output, tuple):
            activation: Float[Tensor, "batch_size seq_len d_model"] = output[0]
        else:
            activation: Float[Tensor, "batch_size seq_len d_model"] = output

        direction = direction / (direction.norm(dim=-1, keepdim=True) + 1e-8)
        direction = direction.to(activation)
        print(direction.shape)
        print(activation.shape)
        # activation -= 2*(activation @ direction).unsqueeze(-1) * direction
        activation -= direction

        if isinstance(output, tuple):
            return (activation, *output[1:])
        else:
            return activation

    return hook_fn

In [ ]:

directions = {}
used_module_names = []
seq_output_file_harmful = OUTPUT_DIR / f"sequential_acts_harmful_{LANGUAGE}_{MODEL_PATH.split('/')[-1]}.npy"
# get activations for harmful instructions then save to file
seq_harmful_acts, cache = get_activations(
    model,
    harmful_inst_train[:N_INST_TRAIN],
    batch_size=BATCH_SIZE,
    act_names=act_names,
    num_last_tokens=num_last_tokens,
)
seq_harmful_acts = harmful_acts.cpu().float()
np.save(seq_output_file_harmful, seq_harmful_acts.numpy())
for l in range(model.cfg.n_layers):
    print(l)
    fwd_hooks = [(f"blocks.{ly}.hook_resid_post", get_direction_ablation_output_hook(ly, directions[ly])) for ly in used_module_names]
    print(fwd_hooks)
        # These hooks will intervene on the model, that's why we need hook_* args.
    # get contranstive activations
    seq_output_file_harmless = OUTPUT_DIR / f"sequential_acts_harmless_{LANGUAGE}_{MODEL_PATH.split('/')[-1]}.npy"
    if len(fwd_hooks)> 0:
        with model.hooks(fwd_hooks=fwd_hooks):   
            # get activations for harmless instructions then save to file
            seq_harmless_acts, cache = get_activations(
                model,
                harmless_inst_train[:N_INST_TRAIN],
                batch_size=BATCH_SIZE,
                act_names=act_names,
                num_last_tokens=num_last_tokens,
            )
            seq_harmless_acts = seq_harmless_acts.cpu().float()
    else:
        seq_harmless_acts, cache = get_activations(
            model,
            harmless_inst_train[:N_INST_TRAIN],
            batch_size=BATCH_SIZE,
            act_names=act_names,
            num_last_tokens=num_last_tokens,
        )
        seq_harmless_acts = seq_harmless_acts.cpu().float()  
    
    seq_harmful_acts_normed = seq_harmful_acts / seq_harmful_acts.norm(dim=-1, keepdim=True)
    seq_harmless_acts_normed = seq_harmless_acts / seq_harmless_acts.norm(dim=-1, keepdim=True)
    
    seq_harmful_acts_normed_mean = seq_harmful_acts_normed.mean(dim=2)
    seq_harmless_acts_normed_mean = seq_harmless_acts_normed.mean(dim=2)
    np.save(seq_output_file_harmless, seq_harmless_acts.numpy())
    
    ref_dir = seq_harmless_acts_normed_mean - seq_harmful_acts_normed_mean
    d_model = ref_dir.shape[-1]
    used_module_names.append(l)
    ref_dir = ref_dir[:, :, -1].reshape(-1, d_model)
    directions[l] = ref_dir[l]


In [ ]:
seq_refusal_dirs = seq_harmless_acts_normed_mean[:, :, chosen_token] - seq_harmful_acts_normed_mean[:, :, chosen_token]
df_norms = plot_vector_norms(seq_refusal_dirs, ord=2.0)

## Refusal direction analysis


### Pairwise cosine of refusal directions at each layer


In [ ]:
layer_names = sum([[f"{i}-mid", f"{i}-post"] for i in range(num_layers)], [])

dirs = refusal_dirs.reshape(-1, refusal_dirs.shape[-1])
A = dirs @ dirs.T

fig = px.imshow(
    # np.rad2deg(np.arccos(np.clip(A, 0.0, 1.0))),
    A,
    x=layer_names,
    y=layer_names,
    width=len(layer_names) * 14,
    height=len(layer_names) * 14,
    title="Cosine Similarity Matrix",
    color_continuous_scale="Viridis",
)
fig.update_layout(
    yaxis=dict(dtick=1),
    xaxis=dict(dtick=1),
)
# fig.update_traces(xgap=1, ygap=1)
fig.show()

### Mean cosine of refusal directions at each layer with at other layers


In [ ]:
from torch.nn.functional import normalize

layer_names = sum([[f"{i}", f"{i}-post"] for i in range(num_layers)], [])
layer_names = [str(i) for i in range(2 * num_layers)]

harmful_acts_normed_mean_normed = normalize(
    harmful_acts_normed_mean[:, :, chosen_token], dim=-1
)
harmless_acts_normed_mean_normed = normalize(
    harmless_acts_normed_mean[:, :, chosen_token], dim=-1
)
raw_dirs = harmful_acts_normed_mean_normed - harmless_acts_normed_mean_normed

raw_dirs = raw_dirs.reshape((-1, raw_dirs.shape[-1]))

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=layer_names,
        y=raw_dirs.norm(dim=-1),
        mode="lines+markers",
        yaxis="y",
        marker_color=colour_map_light["neutral"],
        marker_size=8,
        showlegend=False,
    )
)
fig.add_trace(
    go.Scatter(
        x=layer_names[::],
        y=raw_dirs.norm(dim=-1)[::],
        mode="markers",
        yaxis="y",
        marker_color=colour_map["neutral"],
        marker_size=8,
        showlegend=False,
    )
)

print(layer_names[np.argmax(raw_dirs.norm(dim=-1)[:-1])])


fig.update_layout(
    # title=(
    #     "Statistics of refusal direction candidates at each layer"
    #     f" layer for {MODEL_PATH}"
    # ),
    plot_bgcolor="white",
    grid=dict(rows=1, columns=1),
    xaxis=dict(
        type="category",
        title=dict(text="Extraction Point", font=dict(size=28)),
        dtick=4,
        gridcolor="lightgrey",
        tickfont=dict(size=24),
    ),
    yaxis=dict(
        title=dict(text="Norm of<br>Refusal Direction", font=dict(size=28)),
        gridcolor="lightgrey",
        zeroline=False,
        tickfont=dict(size=24),
    ),
    hovermode="x unified",
    height=300,
    width=1000,
    # width=20 + 12 * len(x_values),
    margin=dict(l=20, r=20, t=20, b=20),
)
fig.show()

fig.write_image(VISUALIZATION_DIR / "norm_refusal.pdf", scale=5)


flatten_dirs = refusal_dirs.reshape(-1, refusal_dirs.shape[-1])
pairwise_cosine = flatten_dirs @ flatten_dirs.T
# pairwise_cosine = np.arccos(pairwise_cosine)
mean_cosine = np.nanmean(pairwise_cosine, axis=-1)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=layer_names,
        y=mean_cosine,
        mode="lines+markers",
        yaxis="y",
        marker_color=colour_map_light["neutral"],
        showlegend=False,
        marker_size=8,
    )
)
fig.add_trace(
    go.Scatter(
        x=layer_names[::],
        y=mean_cosine[::],
        mode="markers",
        yaxis="y",
        marker_color=colour_map["neutral"],
        showlegend=False,
        marker_size=8,
    )
)

# fig.add_trace(
#     go.Scatter(
#         x=layer_names,
#         y=raw_dirs.norm(dim=-1) + mean_cosine / mean_cosine.max(),
#         mode="lines+markers",
#         yaxis="y3",
#         marker_color=colour_map_light["neutral"],
#         showlegend=False
#     )
# )

fig.update_layout(
    # title=(
    #     "Statistics of refusal direction candidates at each extraction point"
    #     f" for {MODEL_PATH}"
    # ),
    plot_bgcolor="white",
    grid=dict(rows=1, columns=1),
    xaxis=dict(
        type="category",
        title=dict(text="Extraction Point", font=dict(size=28)),
        dtick=4,
        gridcolor="lightgrey",
        tickfont=dict(size=24),
    ),
    yaxis=dict(
        title=dict(text=f"Mean<br>Cosine Score", font=dict(size=28)),
        gridcolor="lightgrey",
        zeroline=False,
        tickfont=dict(size=24),
    ),
    hovermode="x unified",
    height=300,
    width=1000,
    # width=20 + 12 * len(x_values),
    margin=dict(l=20, r=20, t=20, b=20),
)

fig.show()
layer_names[np.nanargmax(mean_cosine)]


fig.write_image(VISUALIZATION_DIR / "mean_cosine.pdf", scale=5)

In [ ]:
# sanity check

# layers x resid_modules x tokens x batch x dim
category2acts_normed = {
    "harmful": harmful_acts_normed,
    "harmless": harmless_acts_normed,
}

print(refusal_dirs.shape)
x = category2acts_normed["harmful"][..., chosen_token, :]
print(x.shape)

a = einops.einsum(
    refusal_dirs, x, "layer act dim, layer act batch dim -> layer act batch"
)
# print(a[0][0] - refusal_dirs[0][0] @ x[0][0].T)
# assert np.allclose(a[0][0], refusal_dirs[0][0] @ x[0][0].T, atol=1e-6)

### Scalar projections of activations onto the local refusal direction at each extraction point


In [ ]:
# layers x resid_modules x tokens x batch x dim
category2acts_normed = {
    "harmful": harmful_acts_normed,
    "harmless": harmless_acts_normed,
}

# x_values = sum([[f"{l}", f"{l}-post"] for l in range(num_layers)], [])
x_values = [str(i) for i in range(2 * num_layers)]

fig = go.Figure()

for category in categories:
    acts_normed = category2acts_normed[category][:, :, :, chosen_token]
    projections = einops.einsum(
        refusal_dirs,
        acts_normed,
        "layer act dim, layer act batch dim -> layer act batch",
    )
    projections = torch.tensor(projections)

    mean_projection = projections.mean(dim=-1)

    y_values = mean_projection.flatten()

    # mean
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=y_values,
            name=category,
            mode="lines+markers",
            yaxis="y",
            marker=dict(color=colour_map[category], size=3),
            showlegend=True,
        )
    )
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=y_values,
            name=category,
            mode="lines+markers",
            yaxis="y",
            marker=dict(color=colour_map_light[category], size=3),
            showlegend=False,
        )
    )

    # variance
    fig.add_trace(
        variance_plot(
            x=x_values,
            y=projections.reshape(-1, projections.shape[-1]),
            yaxis="y",
            fillcolor=colour_map_opaque[category],
            showlegend=False,
        )
    )

    # dot markers
    fig.add_trace(
        go.Scatter(
            x=x_values[1::],
            y=y_values[1::],
            name=f"{category}",
            mode="markers",
            yaxis="y",
            marker=dict(color=colour_map[category], size=3),
            showlegend=False,
        )
    )


fig.update_layout(
    # title=f"Scalar projections of activations onto the local refusal direction at each
    # extraction point for {MODEL_PATH}",
    plot_bgcolor="white",
    grid=dict(rows=1, columns=1),
    xaxis=dict(
        type="category",
        dtick=4,
        title=dict(text="Extraction Point", font=dict(size=20)),
        gridcolor="lightgrey",
        tickfont=dict(size=18),
    ),
    yaxis=dict(
        title=dict(text="Scalar Projections", font=dict(size=20)),
        gridcolor="lightgrey",
        zeroline=False,
        tickfont=dict(size=18),
    ),
    hovermode="x unified",
    height=250,
    # width=20 + 12 * len(x_values),
    width=600,
    margin=dict(l=0, r=0, t=0, b=0),
    legend=dict(x=0.05, y=0.95, font=dict(size=18)),
)
fig.show()

fig.write_image(VISUALIZATION_DIR / "prj_onto_local_refusal_candidates.pdf", scale=5)

### Criteria for selecting the refusal direction


In [ ]:
# Criteria: highest norm
criteria = raw_dirs.norm(dim=-1)[:-1]

argmax = np.nanargmax(criteria)
max_norm_layer = argmax // 2
max_norm_act_idx = argmax % 2

print(
    f"Highest refusal direction norm at layer {max_norm_layer}, module"
    f" {act_names[max_norm_act_idx]}, position {chosen_token}"
)

# Criteria: High similiarity
argmax = np.nanargmax(mean_cosine)
max_mean_cosine_layer = argmax // 2
max_mean_cosine_act_idx = argmax % 2

print(
    f"Highest cosine similarity at layer {max_mean_cosine_layer}, module"
    f" {act_names[max_mean_cosine_act_idx]}, position {chosen_token}"
)

### Selecting the refusal direction


In [ ]:
chosen_layer = max_mean_cosine_layer
chosen_act_idx = max_mean_cosine_act_idx
chosen_token = -1

### Projection of activation at each extraction poin onto the chosen refusal direction


In [ ]:
fig = go.Figure()

for category in ["harmful", "harmless"]:
    if category == "harmful":
        # acts_normed = harmful_acts.cpu().float().numpy()
        acts_normed = harmful_acts_normed
    else:
        # acts_normed = harmless_acts.cpu().float().numpy()
        acts_normed = harmless_acts_normed

    # layers x resid_modules x batch_size x dim
    activations = acts_normed[..., chosen_token, :].copy()

    # dim
    direction = refusal_dirs[chosen_layer, chosen_act_idx].copy()

    # layers x resid_modules x batch_size
    scalar_projections = einops.einsum(
        activations,
        direction,
        "... batch_size dim, ... dim -> ... batch_size",
    )
    scalar_projections = np.nan_to_num(scalar_projections)
    print(category)
    print(scalar_projections.mean())
    degrees = np.rad2deg(np.arccos(scalar_projections))

    y_values = scalar_projections

    batch_size = scalar_projections.shape[-1]

    # x_values_flatten = sum(
    #     [
    #         [f"{l}-mid"] * batch_size + [f"{l}-post"] * batch_size
    #         for l in range(num_layers)
    #     ],
    #     [],
    # )
    x_values = sum([[f"{l}", f"{l}-post"] for l in range(num_layers)], [])
    x_values = [str(i) for i in range(2 * num_layers)]

    # variance
    fig.add_trace(
        variance_plot(
            x=x_values,
            y=torch.tensor(y_values).reshape(-1, degrees.shape[-1]),
            yaxis="y",
            fillcolor=colour_map_opaque[category],
            showlegend=False,
        )
    )

    # mean
    ## for legend
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=y_values.mean(axis=-1).flatten(),
            mode="lines+markers",
            yaxis="y",
            marker=dict(color=colour_map[category], size=3),
            showlegend=True,
            name=category,
        )
    )
    ## for lines
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=y_values.mean(axis=-1).flatten(),
            mode="lines",
            yaxis="y",
            marker=dict(color=colour_map_light[category], size=3),
            showlegend=False,
            name=category,
        )
    )
    ## for markers
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=y_values.mean(axis=-1).flatten(),
            mode="markers",
            yaxis="y",
            marker=dict(color=colour_map[category], size=3),
            showlegend=False,
            name=category,
        )
    )

    activations -= 2 * einops.einsum(
        np.maximum(scalar_projections, 0),
        direction,
        "layer resid_module batch_size, dim -> layer resid_module batch_size dim",
    )
    scalar_projections = einops.einsum(
        activations,
        direction,
        "... batch_size dim, ... dim -> ... batch_size",
    )
    print(category)
    print(scalar_projections.mean())
    degrees = np.rad2deg(np.arccos(scalar_projections))

    y_values = scalar_projections


module_names = ["mid", "post"]
fig.update_layout(
    grid=dict(rows=1, columns=1),
    # yaxis=dict(tickformat=".2E"),
    plot_bgcolor="white",
    xaxis=dict(
        type="category",
        dtick=4,
        title=dict(text="Extraction Point", font=dict(size=20)),
        gridcolor="lightgrey",
        tickfont=dict(size=18),
    ),
    yaxis=dict(
        title=dict(text="Scalar Projections", font=dict(size=20)),
        gridcolor="lightgrey",
        zeroline=False,
        tickfont=dict(size=18),
    ),
    hovermode="x unified",
    height=250,
    width=600,
    # title=(
    #     "Scalar projections of activations at each layer onto the chosen refusal direction"
    #     f" ({chosen_layer}-{module_names[chosen_act_idx]})"
    # ),
    # yaxis=dict(matches=None),
    margin=dict(l=20, r=20, t=20, b=20),
    legend=dict(x=0.05, y=0.95, font=dict(size=18)),
)
fig.show()

fig.write_image(VISUALIZATION_DIR / "prj_onto_refusal_dir.pdf", scale=5)

In [ ]:
# sanity check

print(harmful_acts.shape)
a = harmful_acts[chosen_layer, chosen_act_idx, 0, chosen_token]
an = a / a.norm()
an = an.cpu().float()
b = harmful_acts_normed[chosen_layer, chosen_act_idx, 0, chosen_token].copy()

print(an.dtype)
print(b.dtype)

print(an, np.linalg.norm(an))
print(b, np.linalg.norm(b))
np.testing.assert_allclose(an, b, rtol=10e-6)

### Scalar projections of weights at each SelfAttn and MLP layer onto the refusal direction


In [ ]:
layer_names = sum([[f"{i}-mid", f"{i}-post"] for i in range(num_layers)], [])
direction = refusal_dirs[chosen_layer, chosen_act_idx].copy()
direction /= np.linalg.norm(direction)

x_values = []
prj_values = []
sum_magnitude = []

for i in range(num_layers):
    W = model.blocks[i].attn.W_O
    prjs = W.detach().cpu().float().numpy() @ direction
    prjs = prjs.flatten()
    prj_values.append(prjs)
    x_values.extend([layer_names[i * 2]] * prjs.shape[0])
    sum_magnitude.append(np.sum(np.abs(prjs)))

    W = model.blocks[i].mlp.W_out
    prjs = W.detach().cpu().float().numpy() @ direction
    prjs = prjs.flatten()
    prj_values.append(prjs)
    x_values.extend([layer_names[i * 2 + 1]] * prjs.shape[0])
    sum_magnitude.append(np.sum(np.abs(prjs)))

fig = go.Figure()

fig.add_trace(
    go.Box(
        x=x_values,
        y=np.hstack(prj_values),
        boxmean=True,
        marker_color=px.colors.qualitative.Plotly[3],
    )
)
fig.update_layout(
    title=(
        "Scalar projections of weights at each layer onto the refusal direction"
        f" ({chosen_layer}-{module_names[chosen_act_idx]})"
    ),
    yaxis_range=[-0.5, 0.5],
)

fig.show()

fig = px.line(x=layer_names, y=sum_magnitude, markers=True)
fig.show()

## Constructing the steering plane


### Compute the PCA from candidate directions


In [ ]:
refusal_dirs_flatten = refusal_dirs.reshape(-1, refusal_dirs.shape[-1])
refusal_dirs_flatten.shape

from sklearn.decomposition import PCA

pca_model = PCA().fit(refusal_dirs_flatten)

In [ ]:
vars = pca_model.explained_variance_ratio_
vars

In [ ]:
components = pca_model.components_

print(refusal_dirs_flatten.shape)
print(components.shape)
np.degrees(np.arccos(refusal_dirs_flatten @ components[-1]))

# another adhoc attempt to find the best direction is to take the mean of the candidates
mean_d = refusal_dirs_flatten.mean(axis=0)
mean_d /= np.linalg.norm(mean_d)

In [ ]:
# angle between each component and the chosen direction
print(chosen_layer, chosen_act_idx)
np.degrees(np.arccos(components @ refusal_dirs[chosen_layer][chosen_act_idx]))

### Visualize of the candidate directions on the steering plane


In [ ]:
# first basis is the chosen direction (in this case, the one with the highest similarity)
print(max_mean_cosine_layer, max_mean_cosine_act_idx)
u1 = refusal_dirs[max_mean_cosine_layer][max_mean_cosine_act_idx].copy()

# second basis is the first principal component
u2 = components[0].copy()

b1 = u1 / np.linalg.norm(u1)
b2 = u2 - (u2 @ b1) * b1
b2 /= np.linalg.norm(b2)
P = np.outer(b1, b1) + np.outer(b2, b2)

prj_matrix = np.column_stack([b1, b2])
refusal_dirs_mapped = refusal_dirs_flatten @ prj_matrix

fig = go.Figure()

norms = np.linalg.norm(refusal_dirs_mapped)
x = refusal_dirs_mapped[:, 0] / norms
y = refusal_dirs_mapped[:, 1] / norms
angle = np.arctan2(y, x)

for point, label in zip(
    [u1 @ prj_matrix, u2 @ prj_matrix], ["chosen<br>direction", "1st PC"]
):
    fig.add_annotation(
        # hovertext=str(i),
        ax=0,
        ay=0,
        x=point[0],
        y=point[1],
        axref="x",
        ayref="y",
        showarrow=True,
        arrowhead=2,
        arrowwidth=2,
        xanchor="right",
        yanchor="top",
        opacity=0.5,
    )
    fig.add_annotation(
        x=point[0],
        y=point[1],
        text=label,
        font=dict(size=22),
        showarrow=False,
        yshift=30,
        xshift=20,
    )

points = go.Scatter(
    x=refusal_dirs_mapped[:, 0],
    y=refusal_dirs_mapped[:, 1],
    text=[str(i) for i in range(len(refusal_dirs_mapped))],
    mode="markers",
    marker=dict(
        symbol="arrow",
        angle=90 - np.degrees(angle),
        # size=[i + 7 if not np.isnan(i) else 0 for i in norms.flatten()],
        size=20,
        # color=np.linspace(0, refusal_dirs_mapped.shape[0], refusal_dirs_mapped.shape[0]),
        color=[i for i in range(refusal_dirs_mapped.shape[0])],
        # color=color.flatten(),
        # colorscale="Phase",
        showscale=True,
    ),
    name="layers",
    showlegend=True,
)
fig.add_trace(points)

fig.add_annotation(
    xref="paper",
    yref="paper",
    text="Extraction<br>Point",
    font=dict(size=22),
    showarrow=False,
    x=1.17,
    y=-0.15,
    # yshift=20,
    # xshift=20
)


fig.update_layout(
    # plot_bgcolor="white",
    autosize=False,
    # width=800,
    height=600,
    # yaxis_range=[-1.0, 1.0],
    # xaxis_range=[-0.4, 1.4],
    yaxis_scaleanchor="x",
    yaxis_scaleratio=1,
    xaxis_dtick=0.5,
    yaxis_dtick=0.5,
    font=dict(size=22),
    margin=dict(l=0, r=100, t=0, b=75),
    legend=dict(visible=False),
    # xaxis=dict(gridcolor="grey"),
    # yaxis=dict(gridcolor="grey"),
)

fig.show()

fig.write_image(VISUALIZATION_DIR / "steering_plane.pdf", scale=5)

## Steering by rotation


### Rotation utils


In [ ]:
def get_rotation_matrix(degree, basis1, basis2):
    assert len(basis1.shape) == 1
    assert len(basis2.shape) == 1
    assert basis1.shape == basis2.shape

    n = basis1.shape[-1]

    if degree % 360 == 0:
        return np.eye(n)

    # ensure bases are orthonormal
    u = basis1 / np.linalg.norm(basis1)
    v = basis2 - (basis2 @ u) * u
    v /= np.linalg.norm(v)

    theta = np.deg2rad(degree)
    cos_theta = np.cos(theta)
    sin_theta = np.sin(theta)
    # print(cos_theta, sin_theta)

    # rotate counter-clockwise
    R_theta = [[cos_theta, -sin_theta], [sin_theta, cos_theta]]

    uv = np.column_stack([u, v])
    R = np.eye(n) - (np.outer(u, u) + np.outer(v, v)) + uv @ R_theta @ uv.T

    return R


# sanity check
print(chosen_layer, chosen_act_idx)
d = refusal_dirs[chosen_layer][chosen_act_idx].copy()
b1 = components[-1].copy()
b2 = mean_d.copy()

b1 = b1 / np.linalg.norm(b1)
b2 = b2 - (b2 @ b1) * b1
b2 /= np.linalg.norm(b2)
P = np.outer(b1, b1) + np.outer(b2, b2)

deg = np.rad2deg(np.arccos(b1 @ b2))
print(deg)

R = get_rotation_matrix(30, b1, b2)

u = P @ R @ d
u /= np.linalg.norm(u)
v = P @ d
v /= np.linalg.norm(v)

print(np.rad2deg(np.arccos((R @ d) @ d)))
print(np.rad2deg(np.arccos(u @ v)))

In [ ]:
def rotate_to_target(x, target_degree, basis1, basis2):
    assert len(basis1.shape) == 1
    assert len(basis2.shape) == 1
    assert basis1.shape == basis2.shape

    n = basis1.shape[-1]

    # ensure bases are orthonormal
    u = basis1 / np.linalg.norm(basis1)
    v = basis2 - (basis2 @ u) * u
    v /= np.linalg.norm(v)

    theta = np.deg2rad(target_degree)
    cos_theta = np.cos(theta)
    sin_theta = np.sin(theta)

    P = np.outer(u, u) + np.outer(v, v)

    # rotate counter-clockwise
    R_theta = [[cos_theta, -sin_theta], [sin_theta, cos_theta]]

    uv = np.column_stack([u, v])

    rotated_component = uv @ R_theta @ np.array([1, 0])
    Px = x @ P
    scale = np.linalg.norm(Px, axis=-1, keepdims=True)

    result = x - Px + scale * rotated_component

    return result


# sanity check
d = refusal_dirs[chosen_layer][chosen_act_idx].copy()
b1 = components[-1].copy()
b2 = mean_d.copy()
b1 = b1 / np.linalg.norm(b1)
b2 = b2 - (b2 @ b1) * b1
b2 /= np.linalg.norm(b2)
P = np.outer(b1, b1) + np.outer(b2, b2)

rd = rotate_to_target(d, 60, b1, b2)

u = P @ d
u /= np.linalg.norm(u)
deg = np.rad2deg(np.arccos(u @ b1))
R = get_rotation_matrix(60 - deg, b1, b2)
print(R @ d)
print(rd)

print(d.shape)
print(np.linalg.norm(d, axis=-1, keepdims=True) * np.array([1, 2]))
2 * np.array([1, 2])
rotate_to_target(np.random.rand(5, 8, d.shape[0]), 60, b1, b2).shape
print(chosen_layer, chosen_act_idx)

### Statistics of candidate directions on the steering plane


In [ ]:
print(chosen_layer, chosen_act_idx)
d = refusal_dirs[chosen_layer][chosen_act_idx].copy()
b1 = mean_d.copy()
b2 = components[0].copy()

b1 = b1 / np.linalg.norm(b1)
b2 = b2 - (b2 @ b1) * b1
b2 /= np.linalg.norm(b2)

P = np.outer(b1, b1) + np.outer(b2, b2)

print(mean_d)
print(mean_d @ P)

proj = refusal_dirs_flatten @ P
proj_norm = np.linalg.norm(proj, axis=-1)

fig = px.line(x=layer_names, y=proj_norm)
fig.update_layout(
    title="Norm of projections of candidate directions on the steering plane",
)
fig.show()

proj_normed = proj / proj_norm[:, None]
proj_angle = np.rad2deg(np.arccos(proj_normed @ b1))

fig = px.line(x=layer_names, y=proj_angle)
fig.update_layout(
    title=(
        "Angles between the projections of candiate directions and the chosen"
        f" direction on the steering plane ({chosen_layer}-{chosen_act_idx})"
    ),
)
fig.show()
print(chosen_layer, chosen_act_idx)

### Creating the steering config


In [ ]:
from pprint import pprint

from transformers import AutoModel
from accelerate import init_empty_weights

with init_empty_weights():
    hf_model = AutoModel.from_pretrained(MODEL_PATH)

In [ ]:
# Let's look at all the names of the modules in the model so that we know which ones to
# apply steering to (we're interested in the normalization layers before each MLP and
# attention layer)
pprint(list(n for n, m in hf_model.named_modules()))

In [ ]:
import re
from pprint import pprint

target_modules = ["mid", "post"]
# because in the model's architecture, there are no explicit modules for the residual stream (pre/mid/post)
# thus let's use the inputs of layernorm modules as equivalent
# resid-pre = input of input_layernorm
# resid-mid = input of post_attention_layernorm / pre_feedforward_layernorm (gemma)
# resid-post = input of the next input_layernorm
layernorm_modules = ["input_layernorm", "post_attention_layernorm"]
if "gemma" in MODEL_PATH:
    layernorm_modules += ["post_attention_layernorm", "post_feedforward_layernorm"]


mean_d = refusal_dirs_flatten.mean(axis=0)
mean_d /= np.linalg.norm(mean_d)

print(chosen_layer, chosen_act_idx)

# saving various steering configs
for first_direction, first_dir_name in [
    (
        refusal_dirs[max_norm_layer][max_norm_act_idx].copy(),
        f"dir_max_norm_{max_norm_layer}_{target_modules[max_norm_act_idx]}",
    ),
    (
        refusal_dirs[max_mean_cosine_layer][max_mean_cosine_act_idx].copy(),
        f"dir_max_sim_{max_mean_cosine_layer}_{target_modules[max_mean_cosine_act_idx]}",
    ),
    (mean_d.copy(), "dir_mean"),
]:

    second_direction = components[0].copy()

    num_layers = refusal_dirs.shape[0]
    steering_config = {}
    for layer_idx in range(num_layers):
        for module in layernorm_modules:
            if module != "input_layernorm":
                module_name = f"model.layers.{layer_idx}.{module}"
            elif layer_idx < num_layers - 1:
                module_name = f"model.layers.{layer_idx + 1}.{module}"
            else:
                continue

            steering_config[module_name] = {
                "mode": "rotate_to",
                "first_direction": first_direction,
                "second_direction": second_direction,
            }

    # pprint(steering_config)
    output_name = f"{METHOD_PREFIX}steering_config-en-{first_dir_name}-pca_0.npy"
    print(output_name)
    np.save(OUTPUT_DIR / output_name, steering_config)

### Testing Angular Steering

(this takes a while because inference with transformers-lens is slow)


#### Try with your own prompt


In [ ]:
# def get_rotate_to_target_func(target_degree, basis1, basis2):
#     assert len(basis1.shape) == 1
#     assert len(basis2.shape) == 1
#     assert basis1.shape == basis2.shape

#     n = basis1.shape[-1]

#     # ensure bases are orthonormal
#     u = basis1 / np.linalg.norm(basis1)
#     v = basis2 - (basis2 @ u) * u
#     v /= np.linalg.norm(v)

#     theta = np.deg2rad(target_degree)
#     cos_theta = np.cos(theta)
#     sin_theta = np.sin(theta)

#     P = np.outer(u, u) + np.outer(v, v)

#     # rotate counter-clockwise
#     R_theta = [[cos_theta, -sin_theta], [sin_theta, cos_theta]]

#     uv = np.column_stack([u, v])

#     rotated_component = uv @ R_theta @ np.array([1, 0])

#     def __func(x: Tensor):
#         Px = x @ torch.tensor(P, device=x.device, dtype=x.dtype)
#         scale = Px.norm(dim=-1, keepdim=True)

#         result = (
#             x
#             - Px
#             + scale * torch.tensor(rotated_component, device=x.device, dtype=x.dtype)
#         )

#         return result

#     return __func

# transformation_func = get_rotate_to_target_func(
#     target_degree=180, basis1=refusal_dir.copy(), basis2=components[0].copy()
# )

# fwd_hooks = [
#     (
#         utils.get_act_name(act_name, layer),
#         functools.partial(
#             activation_rotation_hook,
#             transformation_func=transformation_func,
#         ),
#     )
#     for layer in intervention_layers
#     for act_idx, act_name in enumerate(["resid_pre", "resid_mid"])
# ]

# answer = get_generations(
#     model,
#     ["how to pick a lock"],
#     model.tokenizer,
#     fwd_hooks=fwd_hooks,
#     max_tokens_generated=256,
# )

# for a in answer:
#     print(a.split(model.tokenizer.eos_token)[0])

#### Steered generation on test set at every 10 steering degree

(very very slow)


In [ ]:
# MAX_GENERATION_TOKENS = 512
# NUM_TEST_SAMPLES = 4
# test_samples = harmful_inst_test[:NUM_TEST_SAMPLES]


# def get_rotate_to_target_func(target_degree, basis1, basis2):
#     assert len(basis1.shape) == 1
#     assert len(basis2.shape) == 1
#     assert basis1.shape == basis2.shape

#     n = basis1.shape[-1]

#     # ensure bases are orthonormal
#     u = basis1 / np.linalg.norm(basis1)
#     v = basis2 - (basis2 @ u) * u
#     v /= np.linalg.norm(v)

#     theta = np.deg2rad(target_degree)
#     cos_theta = np.cos(theta)
#     sin_theta = np.sin(theta)

#     P = np.outer(u, u) + np.outer(v, v)

#     # rotate counter-clockwise
#     R_theta = [[cos_theta, -sin_theta], [sin_theta, cos_theta]]

#     uv = np.column_stack([u, v])

#     rotated_component = uv @ R_theta @ np.array([1, 0])

#     def __func(x: Tensor):
#         Px = x @ torch.tensor(P, device=x.device, dtype=x.dtype)
#         scale = Px.norm(dim=-1, keepdim=True)

#         result = (
#             x
#             - Px
#             + scale * torch.tensor(rotated_component, device=x.device, dtype=x.dtype)
#         )

#         return result

#     return __func


# def activation_rotation_hook(
#     activation: Float[Tensor, "... d_act"],
#     hook: HookPoint,
#     transformation_func,
# ):
#     return transformation_func(activation)


# if not "baseline_generations" in locals():
#     # if True:
#     baseline_generations = get_generations(
#         model,
#         test_samples,
#         model.tokenizer,
#         fwd_hooks=[],
#         max_tokens_generated=MAX_GENERATION_TOKENS,
#     )

# intervention_generations = {}
# refusal_dir = refusal_dirs[chosen_layer][chosen_act_idx]
# for degree in range(0, 360, 10):
#     intervention_layers = list(range(model.cfg.n_layers))
#     print("degree", degree)
#     print(intervention_layers)

#     if degree in intervention_generations:
#         continue

#     transformation_func = get_rotate_to_target_func(
#         target_degree=degree, basis1=refusal_dir.copy(), basis2=components[0].copy()
#     )

#     fwd_hooks = [
#         (
#             utils.get_act_name(act_name, layer),
#             functools.partial(
#                 activation_rotation_hook,
#                 transformation_func=transformation_func,
#             ),
#         )
#         for layer in intervention_layers
#         for act_idx, act_name in enumerate(["resid_pre", "resid_mid"])
#     ]

#     intervention_generations[degree] = get_generations(
#         model,
#         test_samples,
#         model.tokenizer,
#         fwd_hooks=fwd_hooks,
#         max_tokens_generated=MAX_GENERATION_TOKENS,
#     )

# for i in range(NUM_TEST_SAMPLES):
#     print(f"INSTRUCTION {i}: {repr(test_samples[i])}")
#     print(Fore.GREEN + f"BASELINE COMPLETION:")
#     print(
#         textwrap.fill(
#             baseline_generations[i],
#             width=100,
#             initial_indent="\t",
#             subsequent_indent="\t",
#         )
#     )
#     print(Fore.RESET)
#     for degree in sorted(intervention_generations.keys()):
#         print(Fore.RED + f"INTERVENTION COMPLETION (degree {degree}):")
#         print(intervention_generations[degree][i].split(model.tokenizer.eos_token)[0])
#         # print(
#         #     textwrap.fill(
#         #         intervention_generations[extraction_layer][i],
#         #         width=100,
#         #         initial_indent="\t",
#         #         subsequent_indent="\t",
#         #     )
#         # )
#         print(Fore.RESET)